# Lesson 1.3 — experiments: the state-space explosion

Run the cells in order. Before the experiment, write down your prediction, and only then run the code.

The first cell chooses the implementation:

- `"mine"` uses your `task.py`; solve Part B of the task first;
- `"reference"` uses the course's reference implementation. Set `IMPLEMENTATION = "reference"` if you want to run the notebook before finishing the task.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.world import reference as algorithms

The function `plan(k)` builds the parcel pickup problem on the map with aisles with the first `k` parcels from the list, solves it with the reference BFS, and returns the number of expanded nodes and the plan length.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from introai.search.reference import bfs
from introai.trace import TraceLimitExceeded, record
from introai.world.warehouse import WarehouseMap

MAPS = Path.cwd().parents[2] / "data" / "maps"
aisles = WarehouseMap.from_file(MAPS / "aisles.txt")
PARCEL_CELLS = [(3, 1), (5, 4), (9, 21), (13, 15), (1, 20), (7, 13), (13, 8), (11, 23)]
FREE_CELLS = sum(
    aisles.is_free((row, col)) for row in range(aisles.rows) for col in range(aisles.cols)
)


def plan(k, limit=500_000):
    """Solve the problem with the first k parcels; return (nodes expanded, plan length)."""
    problem = algorithms.ParcelPickupProblem(aisles.with_parcels(PARCEL_CELLS[:k]))
    try:
        with record(limit=limit, kinds={"expand"}) as trace:
            node = bfs(problem)
    except TraceLimitExceeded:
        raise RuntimeError(
            f"BFS expanded more than {limit} nodes - check task.py "
            '(or set IMPLEMENTATION = "reference" in the first cell)'
        ) from None
    return len(trace.events), len(node.solution())

## Experiment: from 1 to 8 parcels

**Your prediction.** How many times does the number of BFS expansions grow when you add one parcel? Will the plan length grow just as fast?

*My prediction:* …

In [ ]:
counts = list(range(1, 9))
expansions = []
print(f"{'parcels':>7} {'states':>8} {'expanded':>9} {'plan':>5}")
for k in counts:
    expanded, length = plan(k)
    expansions.append(expanded)
    print(f"{k:>7} {FREE_CELLS * 2**k:>8} {expanded:>9} {length:>5}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(counts, expansions, color="#2a78d6", linewidth=2, marker="o", markersize=8)
ax.set_title("Nodes expanded by BFS for 1 to 8 parcels", loc="left", color="#0b0b0b")
ax.set_xlabel("number of parcels", color="#52514e")
ax.set_ylabel("nodes expanded", color="#52514e")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
ax.annotate(
    f"{expansions[-1]:,}",
    (counts[-1], expansions[-1]),
    textcoords="offset points",
    xytext=(-8, 6),
    ha="right",
    color="#0b0b0b",
)
fig.tight_layout()

**Think about it.** Compare the result with your prediction. How many expansions does each extra parcel add? Why does the plan length grow much more slowly than BFS's work? What would happen with 20 parcels? Module 2 shows smarter search, and module 4 approximate methods.